In [1]:
from pyspark.sql import SparkSession

# Test session creation
spark = SparkSession.builder \
    .appName("Test_Session") \
    .getOrCreate()

print("Success! Spark version running:", spark.version)

C:\Users\izhan\anaconda3\envs\spark-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Success! Spark version running: 4.2.0


In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum as spark_sum, to_timestamp, min, max, mean

# Initialize the Spark Session
spark = SparkSession.builder \
    .appName("Ecommerce_Week5_Analysis") \
    .getOrCreate()

# Define the folder path
folder_path = r"D:\CELEBAL_CEI_IZHAN\WEEK5_Celebal\ecommerce dataset"

# Automatically find the CSV file in that folder
files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
if not files:
    raise FileNotFoundError(f"No CSV file found in {folder_path}.")

file_name = files[0]
path_to_file = os.path.join(folder_path, file_name)

# Load the dataset
df = spark.read.csv(path_to_file, header=True, inferSchema=True)
print(f"Loaded: {file_name} successfully!")
df.show(3)

Loaded: Ecommerce.csv successfully!
+-----------+----------+----------+-----------+---------+-----------------+----------+----------------+----------+--------+----------------+---------------+-------+------------+----------------+-------------+---------+--------------+------+-----------+--------------------+--------------+---------+-----------+-------------+------------+-----------------------+------------------+--------+
|customer_id|session_id|visit_date|device_type|user_type|marketing_channel|product_id|product_category|unit_price|quantity|discount_percent|discount_amount|revenue|pages_viewed|time_on_site_sec|added_to_cart|purchased|cart_abandoned|rating|review_text|review_helpful_votes|payment_method|visit_day|visit_month|visit_weekday|visit_season|session_duration_bucket|revenue_normalized|location|
+-----------+----------+----------+-----------+---------+-----------------+----------+----------------+----------+--------+----------------+---------------+-------+------------+-------

In [9]:
# Q4: Given a DataFrame df_sales, write a query to filter for rows where 

from pyspark.sql.types import StringType

q4_result = (df.filter(col("location").cast(StringType()) == "West") 
             .groupBy("product_category")
             .agg(avg("revenue").alias("avg_sale_amount")))

q4_result.show(5)

+----------------+---------------+
|product_category|avg_sale_amount|
+----------------+---------------+
+----------------+---------------+



In [7]:
# Q5: What is the difference between .na.drop() and .na.fill()? Provide a code 

df_filled_status = df.na.fill({"payment_method": "Unknown"})
df_filled_status.select("payment_method").distinct().show(5)

+--------------+
|payment_method|
+--------------+
|             1|
|             3|
|             5|
|             4|
|             2|
+--------------+
only showing top 5 rows


In [10]:
# Q6: Write a query to find the total count of records for each city in a DataFrame

q6_result = (df.groupBy("location")
             .agg(count("*").alias("record_count"))
             .filter(col("record_count") > 100))

q6_result.show()

+--------+------------+
|location|record_count|
+--------+------------+
|     148|         104|
|      31|         121|
|      85|         109|
|     137|         113|
|      65|         105|
|      53|         103|
|     133|         113|
|      78|         114|
|     108|         102|
|     211|         124|
|      34|         111|
|     193|         127|
|     126|         115|
|     101|         102|
|      81|         127|
|     210|         103|
|     183|         139|
|      28|         115|
|      76|         117|
|      27|         112|
+--------+------------+
only showing top 20 rows


In [11]:
# Q7: How does the immutability of Spark DataFrames affect how you perform

df_cleaned = df.drop("session_id").withColumnRenamed("revenue", "total_price")
df_cleaned.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- visit_date: string (nullable = true)
 |-- device_type: integer (nullable = true)
 |-- user_type: integer (nullable = true)
 |-- marketing_channel: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_category: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- total_price: double (nullable = true)
 |-- pages_viewed: integer (nullable = true)
 |-- time_on_site_sec: integer (nullable = true)
 |-- added_to_cart: integer (nullable = true)
 |-- purchased: integer (nullable = true)
 |-- cart_abandoned: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- review_text: integer (nullable = true)
 |-- review_helpful_votes: integer (nullable = true)
 |-- payment_method: integer (nullable = true)
 |-- visit_day: integer (nullable = true)
 |-- 

In [12]:
# Q10: Write the code to revise a column named raw_timestamp by casting 

df_timestamped = df.withColumn("event_time", to_timestamp(col("visit_date"), "yyyy-MM-dd"))
df_timestamped.select("visit_date", "event_time").printSchema()

root
 |-- visit_date: string (nullable = true)
 |-- event_time: timestamp (nullable = true)



In [13]:
# Q12: Write a code snippet that identifies and removes rows where the email 
# column contains null values OR the username is an empty string.

df_valid_users = df.filter(col("customer_id").isNotNull() & (col("payment_method") != ""))
print("Cleaned rows syntax applied successfully.")

Cleaned rows syntax applied successfully.


In [14]:
# Q13: How do you use the .agg() function to calculate multiple statistics 

q13_result = df.agg(
    min("unit_price").alias("min_price"),
    max("unit_price").alias("max_price"),
    mean("unit_price").alias("avg_price")
)

q13_result.show()

+---------+---------+----------------+
|min_price|max_price|       avg_price|
+---------+---------+----------------+
|    50.05|  1999.83|782.319010400002|
+---------+---------+----------------+



In [15]:
# Q15: Write a final processing pipeline that:
# 1. Filters out duplicates.
# 2. Fills null prices with 0.
# 3. Groups by store_id to calculate total revenue.

final_pipeline = (df
    .dropDuplicates()
    .na.fill(0, subset=["unit_price", "revenue"])
    .groupBy("customer_id")
    .agg(spark_sum("revenue").alias("total_revenue"))
)

final_pipeline.show(10)

+-----------+-----------------+
|customer_id|    total_revenue|
+-----------+-----------------+
|       6466|              0.0|
|       5803|              0.0|
|       8638|          1292.67|
|       7754|              0.0|
|       1645|          3067.74|
|       4900|          1777.66|
|       7253|              0.0|
|       6658|4624.110000000001|
|       9900|              0.0|
|       3997|              0.0|
+-----------+-----------------+
only showing top 10 rows


In [3]:
import os
from pyspark.sql import SparkSession

# 1. Initialize or get the existing Spark Session
spark = SparkSession.builder \
    .appName("Ecommerce_Week5_Analysis") \
    .getOrCreate()

# 2. Path to your dataset directory or file
folder_path = r"D:\CELEBAL_CEI_IZHAN\WEEK5_Celebal\ecommerce dataset"

# 3. Read the CSV with header and inferSchema enabled
df = spark.read.csv(folder_path, header=True, inferSchema=True)

# 4. Verify data loading
df.show(5)

C:\Users\izhan\anaconda3\envs\spark-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Py4JJavaError: An error occurred while calling o30.csv.
: java.lang.UnsatisfiedLinkError: 'boolean org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(java.lang.String, int)'
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access0(Native Method)
	at org.apache.hadoop.io.nativeio.NativeIO$Windows.access(NativeIO.java:817)
	at org.apache.hadoop.fs.FileUtil.canRead(FileUtil.java:1415)
	at org.apache.hadoop.fs.FileUtil.list(FileUtil.java:1620)
	at org.apache.hadoop.fs.RawLocalFileSystem.listStatus(RawLocalFileSystem.java:802)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2079)
	at org.apache.hadoop.fs.FileSystem.listStatus(FileSystem.java:2123)
	at org.apache.hadoop.fs.ChecksumFileSystem.listStatus(ChecksumFileSystem.java:1020)
	at org.apache.spark.util.HadoopFSUtils$.listLeafFiles(HadoopFSUtils.scala:218)
	at org.apache.spark.util.HadoopFSUtils$.$anonfun$parallelListLeafFilesInternal$1(HadoopFSUtils.scala:132)
	at scala.collection.immutable.List.map(List.

In [4]:
import os

# Point directly to the specific CSV file rather than the directory
csv_file_path = r"D:\CELEBAL_CEI_IZHAN\WEEK5_Celebal\ecommerce dataset\Ecommerce.csv"

df = spark.read.csv(csv_file_path, header=True, inferSchema=True)
df.show(5)

+-----------+----------+----------+-----------+---------+-----------------+----------+----------------+----------+--------+----------------+---------------+-------+------------+----------------+-------------+---------+--------------+------+-----------+--------------------+--------------+---------+-----------+-------------+------------+-----------------------+------------------+--------+
|customer_id|session_id|visit_date|device_type|user_type|marketing_channel|product_id|product_category|unit_price|quantity|discount_percent|discount_amount|revenue|pages_viewed|time_on_site_sec|added_to_cart|purchased|cart_abandoned|rating|review_text|review_helpful_votes|payment_method|visit_day|visit_month|visit_weekday|visit_season|session_duration_bucket|revenue_normalized|location|
+-----------+----------+----------+-----------+---------+-----------------+----------+----------------+----------+--------+----------------+---------------+-------+------------+----------------+-------------+---------+--

In [7]:
from pyspark.sql.functions import col

# Explicitly cast to string if product_category holds string values
q5_result = df.filter(col("product_category").cast("string") == "Electronics") \
              .select("product_id", col("unit_price").alias("price"))

q5_result.show(5)

+----------+-----+
|product_id|price|
+----------+-----+
+----------+-----+



In [11]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

df_revised = df.withColumn("price", col("unit_price").cast(DoubleType()))

# Verify
df_revised.select("unit_price", "price").show(5)

+----------+-------+
|unit_price|  price|
+----------+-------+
|    651.57| 651.57|
|    945.27| 945.27|
|    400.44| 400.44|
|   1268.54|1268.54|
|    880.81| 880.81|
+----------+-------+
only showing top 5 rows


In [12]:
from pyspark.sql.functions import col

# Filter for completed orders with revenue > 1000
df_filtered_orders = df.filter(
    (col("purchased") == 1) & (col("revenue") > 1000)
)

# Display the output
df_filtered_orders.select("customer_id", "purchased", "revenue", "unit_price").show(5)

+-----------+---------+-------+----------+
|customer_id|purchased|revenue|unit_price|
+-----------+---------+-------+----------+
|       4949|        1|2410.23|   1268.54|
|       1773|        1|2495.32|    623.83|
|       1829|        1| 1398.4|     437.0|
|       4190|        1|1485.54|    464.23|
|       5010|        1|1058.02|    1113.7|
+-----------+---------+-------+----------+
only showing top 5 rows


In [13]:
from pyspark.sql.functions import col

# Add final_price column with 18% tax applied on unit_price
df_with_tax = df.withColumn("final_price", col("unit_price") * 1.18)

# Verify the result
df_with_tax.select("product_id", "unit_price", "final_price").show(5)

+----------+----------+------------------+
|product_id|unit_price|       final_price|
+----------+----------+------------------+
|       894|    651.57|          768.8526|
|       844|    945.27|         1115.4186|
|       865|    400.44|472.51919999999996|
|       851|   1268.54|         1496.8772|
|       794|    880.81|1039.3557999999998|
+----------+----------+------------------+
only showing top 5 rows


In [16]:
from pyspark.sql.functions import col

# 1. Filter out null customer_ids
df_clean = df.filter(col("customer_id").isNotNull())

# 2. Convert to Pandas and write to CSV
df_clean.toPandas().to_csv("data/output_cleaned_csv.csv", index=False)

print("Data successfully filtered and saved to CSV via Pandas!")

C:\Users\izhan\anaconda3\envs\spark-env\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Data successfully filtered and saved to CSV via Pandas!


In [18]:
from pyspark.sql.functions import col

# Example using actual dataset columns with OR (|) logic
q14_result = df.filter((col("marketing_channel") == 2) | (col("device_type") == 1))

# Display top 5 rows
q14_result.select("customer_id", "marketing_channel", "device_type").show(5)

+-----------+-----------------+-----------+
|customer_id|marketing_channel|device_type|
+-----------+-----------------+-----------+
|       1803|                2|          2|
|       6890|                0|          1|
|       4949|                2|          1|
|       4896|                5|          1|
|       8726|                4|          1|
+-----------+-----------------+-----------+
only showing top 5 rows
